# 전력수급 API 실습: 받아서 · 쌓고 · 계산하기

한국전력거래소가 공공데이터포털(OpenAPI)로 열어 둔 **5분 단위 전력수급**을 직접 받아 봅니다.
지난 세션이 "만들어진 대시보드가 어떻게 돌아가는가"였다면, 이번에는 **그 재료(Parquet)를 누가 매일 사다 놓는가**를 만듭니다.

> **오늘의 핵심 한 줄**  이 API는 '오늘'만 알려 줍니다. 어제 값은 다시 받을 수 없으니, **매일 받아서 쌓아 두는 일**을 자동화해야 합니다.

| 파트 | 내용 | 시간 |
|---|---|---|
| 1 | 인증키로 첫 호출, 응답 읽기 | 15분 |
| 2 | 오늘 하루치를 표로 만들고 검증·차트 | 20분 |
| 3 | 파일(Parquet)로 쌓기, 중복 없이 | 10분 |
| 4 | SQL로 계산하기 (pandas와 나란히) | 15분 |
| 5 | GitHub Actions로 자동화, 대시보드 열기 | 15분 |
| 마무리 | 이 API의 한계, 정리 | 5분 |

**오늘 쓰는 API** (공공데이터포털 · 한국전력거래소)

| API | 돌려주는 것 | 응답 형식 | 쓰는 곳 |
|---|---|---|---|
| 오늘전력수급현황조회_GW | 오늘 0시부터 지금까지 5분 단위 | XML · JSON | Part 1~5 |

## 준비: 인증키 받기

1. [공공데이터포털](https://www.data.go.kr) 회원가입 · 로그인
2. 아래 API 페이지에서 **활용신청** (개발계정은 자동 승인, 하루 100회까지 호출)
   * [「한국전력거래소_오늘전력수급현황조회_GW」](https://www.data.go.kr/data/15158703/openapi.do)
3. 마이페이지 → **인증키 발급현황** → **일반 인증키(Decoding)** 복사

> 신청 직후에는 키가 아직 등록되지 않았다는 오류가 날 수 있습니다. 세션 전날까지 신청해 두세요.
> **키는 비밀번호입니다.** 코딩 창에 직접 적지 말고 아래 입력창에 붙여넣으세요.
> `print(r.url)` 처럼 요청 주소를 찍으면 키가 함께 보이니 공유할 파일에서는 쓰지 마세요.

In [1]:
# 필요한 도구를 불러옵니다. (Colab에는 대부분 있고, duckdb만 설치합니다)
!pip -q install duckdb

import getpass, json, os
from io import StringIO
from urllib.parse import unquote

import duckdb
import pandas as pd
import plotly.express as px
import requests

# 인증키를 입력창에 붙여넣습니다. (화면에 보이지 않고, 현재 작업중인 이 파일에도 저장되지 않습니다)
KEY = unquote(os.environ.get("KPX_API_KEY") or getpass.getpass("공공데이터포털 인증키(Decoding): ").strip())
print("키 길이:", len(KEY))

공공데이터포털 인증키(Decoding): ··········
키 길이: 88


## Part 1. 첫 호출: 오늘의 전력수급 몇 줄만 받아 보기

API 호출은 **주소 + 요청변수**를 보내고 **응답**을 받는 일입니다. 브라우저 주소창에 넣는 것과 같은데, 파이썬이 대신 합니다.
응답의 모양부터 보려고 **3줄만** 받아 봅니다.

| 요청변수 | 뜻 |
|---|---|
| `serviceKey` | 내 인증키 |
| `dataType` | 응답 형식 (`json` 또는 `xml`) |
| `pageNo` | 몇 번째 쪽 |
| `numOfRows` | 한 쪽에 몇 줄 |

In [2]:
URL_TODAY = "https://apis.data.go.kr/B552115/Sukub5mToday/getSukub5mToday"

r = requests.get(URL_TODAY, params={"serviceKey": KEY, "dataType": "json", "pageNo": 1, "numOfRows": 3}, timeout=30)
print(r.status_code)
print(r.text[:1000])

200
{  "response" : {    "header" : {      "resultCode" : "00",      "resultMsg" : "OK"    },    "body" : {      "dataType" : "JSON",      "totalCount" : "224",      "numOfRows" : "3",      "pageNo" : "1",      "items" : {        "item" : [ {          "suppReservePwr" : 32355.90040,          "suppAbility" : 88011.10160,          "baseDatetime" : "20261101000000",          "operReservePwr" : 11724.40040,          "suppReserveRate" : 58.136200000,          "forecastLoad" : 60900,          "rn" : 1,          "operReserveRate" : 20.668600000,          "currPwrTot" : 55655.19920        }, {          "suppReservePwr" : 31500.59960,          "suppAbility" : 86956.29690,          "baseDatetime" : "20261101000500",          "operReservePwr" : 11847.79980,          "suppReserveRate" : 56.803100000,          "forecastLoad" : 60900,          "rn" : 2,          "operReserveRate" : 20.958900000,          "currPwrTot" : 55455.69920        }, {          "suppReservePwr" : 31941.40040,          "suppAb

### 응답 읽는 법

응답은 크게 **header**(성공했나?)와 **body**(내용)로 나뉘고, 내용은 `body → items → item` 안에 있습니다.

| 항목 | 뜻 | 단위 |
|---|---|---|
| `baseDatetime` | 기준일시 | 년월일시분초 |
| `currPwrTot` | 현재수요 | MW |
| `suppAbility` | 공급능력 | MW |
| `forecastLoad` | 최대예측수요 | MW |
| `suppReservePwr` | 공급예비력 | MW |
| `suppReserveRate` | 공급예비율 | % |
| `operReservePwr` | 운영예비력 | MW |
| `operReserveRate` | 운영예비율 | % |
| `rn` | 순번 | |

In [3]:
js = r.json()
root = js.get("response", js)          # 'response' 껍데기가 있을 수도, 없을 수도 있어요
print("header:", root["header"])

item = root["body"]["items"]["item"]   # 한 건이면 dict, 여러 건이면 list (지금은 3건이라 list)
pd.json_normalize(item)                # 어느 쪽이든 표로 바뀝니다

header: {'resultCode': '00', 'resultMsg': 'OK'}


,suppReservePwr,suppAbility,baseDatetime,operReservePwr,suppReserveRate,forecastLoad,rn,operReserveRate,currPwrTot
0,32355.9004,88011.1016,20261101000000,11724.4004,58.1362,60900,1,20.6686,55655.1992
1,31500.5996,86956.2969,20261101000500,11847.7998,56.8031,60900,2,20.9589,55455.6992
2,31941.4004,86966.2031,20261101001000,12237.9004,58.0493,60900,3,21.8162,55024.6992


> 경로가 맞지 않아 `KeyError` 가 나면 `print(js)` 로 전체를 찍어서 눈으로 길을 따라가 보세요. 응답의 모양을 읽는 것이 API 실습의 절반입니다.

### 연습 1-A. 일부러 틀려 보기

키를 틀리게 보내면 어떤 응답이 올까요? 직접 불러서 읽어 보세요. (하루 100회 한도에 포함되니 한 번만!)

In [4]:
bad = requests.get(URL_TODAY, params={"serviceKey": "wrong-key", "dataType": "json", "pageNo": 1, "numOfRows": 3}, timeout=30)
print(bad.status_code)
print(bad.text[:500])

403
{
  "OpenAPI_ServiceResponse": {
    "cmmMsgHeader": {
      "errMsg": "SERVICE_KEY_IS_NOT_REGISTERED_ERROR",
      "returnAuthMsg": "등록되지 않은 서비스키",
      "returnReasonCode": "30"
    }
  }
}



<details><summary>정리</summary>

* **오류도 '응답'으로 옵니다.** 게이트웨이 오류는 `status_code` 가 200이 아닐 수도 있고, 본문에 오류 코드가 담겨 올 수도 있습니다.
  (인증키 오류는 보통 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR` 같은 문구로 옵니다.)
* 그래서 자동화 스크립트는 **"받은 내용이 정말 기대한 모양인가"**를 반드시 확인합니다. (Part 2 검증, Part 5에서 다시 만납니다)

</details>

## Part 2. 오늘 하루치를 표로

「오늘전력수급현황조회」는 **오늘 0시부터 지금까지 5분 간격**으로 돌려줍니다. 하루는 5분 × 288칸이라 `numOfRows=400` 이면 한 번에 다 받습니다.
Part 1 에서는 JSON 으로 봤지만, 여기서는 **XML** 로 받아 표로 바꿉니다. (JSON과 모양만 다르고 내용은 같습니다.)

In [5]:
r = requests.get(URL_TODAY, params={"serviceKey": KEY, "dataType": "xml", "pageNo": 1, "numOfRows": 400}, timeout=30)
print(r.status_code)
print(r.text[:900])

200
<?xml version='1.0' encoding='UTF-8'?><response><header><resultCode>00</resultCode><resultMsg>OK</resultMsg></header><body><dataType>XML</dataType><totalCount>224</totalCount><numOfRows>400</numOfRows><pageNo>1</pageNo><items><item><suppReservePwr>32355.90040</suppReservePwr><suppAbility>88011.10160</suppAbility><baseDatetime>20261101000000</baseDatetime><operReservePwr>11724.40040</operReservePwr><suppReserveRate>58.136200000</suppReserveRate><forecastLoad>60900</forecastLoad><rn>1</rn><operReserveRate>20.668600000</operReserveRate><currPwrTot>55655.19920</currPwrTot></item><item><suppReservePwr>31500.59960</suppReservePwr><suppAbility>86956.29690</suppAbility><baseDatetime>20261101000500</baseDatetime><operReservePwr>11847.79980</operReservePwr><suppReserveRate>56.803100000</suppReserveRate><forecastLoad>60900</forecastLoad><rn>2</rn><operReserveRate>20.958900000</operReserveRate><curr


**키가 아직 안 되는 분만** 아래 셀을 실행하세요. 명세의 항목 이름을 따라 만든 **연습용 가짜 응답**입니다(실제 값이 아닙니다). 이어지는 셀이 그대로 돕니다.
(Part 1은 제 화면으로 함께 보고, 키가 되면 다시 해 보세요!)

In [ ]:
# ⚠️ 키가 아직 승인되지 않은 분만 실행하세요. 연습용 가짜 응답입니다!!
import math
from datetime import datetime, timedelta

def _fake_item(i, day0):
    t = day0 + timedelta(minutes=5 * i)
    demand = 62000 + 14000 * math.sin((i / 12 - 6) / 24 * 2 * math.pi)
    supply = 92000
    f = {"baseDatetime": t.strftime("%Y%m%d%H%M%S"), "currPwrTot": f"{demand:.1f}", "suppAbility": f"{supply:.1f}",
         "forecastLoad": "77000.0", "suppReservePwr": f"{supply - demand:.1f}",
         "suppReserveRate": f"{(supply - demand) / demand * 100:.2f}",
         "operReservePwr": f"{supply - demand - 5000:.1f}", "operReserveRate": f"{(supply - demand - 5000) / demand * 100:.2f}"}
    return "<item>" + "".join(f"<{k}>{v}</{k}>" for k, v in f.items()) + "</item>"

_day0 = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
_n = int((datetime.now() - _day0).total_seconds() // 300)
class r:       # requests 응답처럼 .text 만 있으면 아래 셀이 그대로 동작합니다
    text = "<response><header><resultCode>00</resultCode></header><body><items>" + "".join(_fake_item(i, _day0) for i in range(_n)) + "</items></body></response>"

### XML → 표(DataFrame)

`pandas.read_xml` 이 응답 안의 `<item>` 하나하나를 한 행으로 바꿔 줍니다. 그러면 우리가 아는 표(DataFrame)가 됩니다.

In [6]:
raw = pd.read_xml(StringIO(r.text), xpath=".//item", parser="etree", dtype=str)
print(raw.shape)
raw.head()

(224, 9)


,suppReservePwr,suppAbility,baseDatetime,operReservePwr,suppReserveRate,forecastLoad,rn,operReserveRate,currPwrTot
0,32355.90040,88011.10160,20261101000000,11724.40040,58.136200000,60900,1,20.668600000,55655.19920
1,31500.59960,86956.29690,20261101000500,11847.79980,56.803100000,60900,2,20.958900000,55455.69920
2,31941.40040,86966.20310,20261101001000,12237.90040,58.049300000,60900,3,21.816200000,55024.69920
3,32251.19920,86946.00000,20261101001500,12075.79980,58.965700000,60900,4,21.655100000,54694.80080
4,32653.80080,86918.10160,20261101002000,12309.09960,60.175500000,60900,5,22.246200000,54264.30080


### 이름표 붙이고 자료형 바꾸기

* 영문 항목 이름을 알아보기 쉬운 열 이름으로 바꿉니다. (MW, % 는 이름에 남겨 둡니다)
* 기준일시는 '년월일시분초'라서 **숫자만 남긴 뒤** 날짜·시각으로 바꿉니다. 형식이 `20261031143000` 이든 `2026-10-31 14:30:00` 이든 읽힙니다.
* 나머지는 숫자로 바꿉니다. 바뀌지 않는 값은 `NaN` 이 됩니다.

In [7]:
NAMES = {
    "baseDatetime": "ts",                 # 기준일시
    "currPwrTot": "demand_mw",            # 현재수요
    "suppAbility": "supply_mw",           # 공급능력
    "forecastLoad": "forecast_mw",        # 최대예측수요
    "suppReservePwr": "reserve_mw",       # 공급예비력
    "suppReserveRate": "reserve_pct",     # 공급예비율(%)
    "operReservePwr": "oper_reserve_mw",  # 운영예비력
    "operReserveRate": "oper_reserve_pct" # 운영예비율(%)
}

df = raw[list(NAMES)].rename(columns=NAMES)

digits = df["ts"].astype(str).str.replace(r"\D", "", regex=True).str.ljust(14, "0").str[:14]
df["ts"] = pd.to_datetime(digits, format="%Y%m%d%H%M%S")
for c in df.columns.drop("ts"):
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values("ts").reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ts                224 non-null    datetime64[ns]
 1   demand_mw         224 non-null    float64       
 2   supply_mw         224 non-null    float64       
 3   forecast_mw       224 non-null    int64         
 4   reserve_mw        224 non-null    float64       
 5   reserve_pct       224 non-null    float64       
 6   oper_reserve_mw   224 non-null    float64       
 7   oper_reserve_pct  224 non-null    float64       
dtypes: datetime64[ns](1), float64(6), int64(1)
memory usage: 14.1 KB


### 믿기 전에 확인하기 (검증)

받은 값을 그대로 믿고 쌓으면, 한 번 틀린 값이 몇 달 뒤 그래프에서 발견됩니다. **쌓기 전에 점검하는 함수**를 만들어 둡니다.

In [8]:
def check(df):
    gaps = df["ts"].diff().dropna()
    print("행 수            :", len(df))
    print("시각 범위        :", df["ts"].min(), "~", df["ts"].max())
    print("중복된 시각      :", int(df["ts"].duplicated().sum()))
    print("5분이 아닌 간격  :", int((gaps != pd.Timedelta(minutes=5)).sum()))
    print("수요가 비었거나 0:", int((df["demand_mw"].isna() | (df["demand_mw"] <= 0)).sum()))
    print("수요 범위(MW)    :", df["demand_mw"].min(), "~", df["demand_mw"].max())

check(df)

행 수            : 224
시각 범위        : 2026-11-01 00:00:00 ~ 2026-11-01 18:35:00
중복된 시각      : 0
5분이 아닌 간격  : 0
수요가 비었거나 0: 0
수요 범위(MW)    : 49370.3984 ~ 61913.3984


### 그래프로 보기

숫자만 보면 이상한 곳을 놓치기 쉽습니다. 현재수요와 공급능력을 겹쳐 그려 봅니다.

In [9]:
px.line(df, x="ts", y=["demand_mw", "supply_mw"], title="오늘의 5분 단위 전력수급 (MW)")

> **AI와 함께 일할 때** API 코드를 Claude에게 맡기려면, 말로 설명하기보다 **API 명세(요청변수·응답항목)와 응답 한 건**을 그대로 붙여 주세요. 항목 이름과 단위가 정확히 전달돼서 코드가 훨씬 정확해집니다. 그리고 어떤 도구가 쓴 코드든 위의 `check()` 같은 **검증은 사람이 정합니다.**

## Part 3. 파일로 쌓기

받은 표를 **Parquet** 파일로 저장합니다. (지난 세션: Parquet = 계산 재료, 작고 빠른 표 파일)

In [10]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
PATH = Path("data/supply_5min.parquet")
df.to_parquet(PATH, index=False)

print(PATH.stat().st_size, "bytes")
pd.read_parquet(PATH).shape

16607 bytes


(224, 8)

**같은 시각(`ts`)은 새로 받은 값으로 덮어쓰기**로 해결합니다. (값이 나중에 정정될 수도 있으니 새 값이 우선입니다)

In [11]:
def upsert(old, new):
    """같은 시각(ts)은 새 값으로 덮어쓰고, 시각 순으로 정렬합니다."""
    both = new if old is None else pd.concat([old, new])
    return both.drop_duplicates("ts", keep="last").sort_values("ts").reset_index(drop=True)

merged = upsert(pd.read_parquet(PATH), df)
print("행 수:", len(merged), "/ 중복된 시각:", int(merged["ts"].duplicated().sum()))

행 수: 224 / 중복된 시각: 0


## Part 4. SQL로 계산하기

지난 세션에서 본 대로 **SQL은 DB에게 보내는 주문서이고, pandas와 같은 일을 합니다.** DuckDB는 파이썬 표(DataFrame)를 이름만 부르면 바로 읽습니다.

며칠치가 쌓여야 재미있는 계산이 됩니다. 며칠 쌓아 둔 파일 주소를 알려 주면 `DATA_URL` 에 넣으세요. 비워 두면 Part 3에서 만든 `merged` 를 씁니다.

In [14]:
DATA_URL = "https://jungeunpark-125.github.io/energy_dashboard/data/supply_5min.parquet"

supply = pd.read_parquet(DATA_URL) if DATA_URL else merged
print(supply["ts"].dt.date.nunique(), "일,", len(supply), "행")

2 일, 389 행


In [15]:
print(supply.groupby(supply["ts"].dt.date).size())
print(supply["ts"].min(), "~", supply["ts"].max())

gap = supply["ts"].sort_values().diff()
print(gap.value_counts().head())
print(supply.loc[gap > "5min", "ts"])
print(supply["ts"].duplicated().sum())

ts
2026-10-31    159
2026-11-01    230
dtype: int64
2026-10-31 00:00:00 ~ 2026-11-01 19:05:00
ts
0 days 00:05:00    387
0 days 10:50:00      1
Name: count, dtype: int64
159   2026-11-01
Name: ts, dtype: datetime64[ms]
0


### pandas와 SQL, 같은 질문 두 가지 답

**질문: 하루 중 가장 전기를 많이 쓴 순간 5개는?**

In [16]:
# pandas
supply.nlargest(5, "demand_mw")[["ts", "demand_mw", "supply_mw", "reserve_pct"]]

,ts,demand_mw,supply_mw,reserve_pct
378,2026-11-01 18:15:00,61913.3984,90201.1016,45.6893
381,2026-11-01 18:30:00,61877.0000,90784.7031,46.7181
382,2026-11-01 18:35:00,61851.3984,90750.0000,46.7227
380,2026-11-01 18:25:00,61782.3008,90218.7969,46.0269
379,2026-11-01 18:20:00,61780.3008,90192.0000,45.9881


In [ ]:
# SQL (표 이름은 변수 이름 supply 를 그대로 씁니다)
duckdb.sql("""
SELECT ts, demand_mw, supply_mw, reserve_pct
FROM supply
ORDER BY demand_mw DESC
LIMIT 5
""").df()

,ts,demand_mw,supply_mw,reserve_pct
0,2026-10-31 00:00:00,59793.5000,90767.1016,51.8010
1,2026-10-31 00:05:00,59384.8984,91071.8984,53.3588
2,2026-10-31 00:10:00,59084.1992,91084.3984,54.1603
3,2026-10-31 00:15:00,58650.6016,91093.7969,55.3162
4,2026-10-31 00:20:00,58346.3984,91019.0000,55.9976


**질문: 날짜별 최대수요는?**  (`GROUP BY` = pandas의 `groupby`)

In [17]:
supply.groupby(supply["ts"].dt.date)["demand_mw"].max()

,demand_mw
ts,
2026-10-31,59793.5000
2026-11-01,61913.3984


In [18]:
duckdb.sql("""
SELECT CAST(ts AS DATE) AS 날짜, max(demand_mw) AS 최대수요_MW
FROM supply
GROUP BY 1
ORDER BY 1
""").df()

,날짜,최대수요_MW
0,2026-10-31,59793.5000
1,2026-11-01,61913.3984


## 마무리: 이 API의 한계

* 이 값은 5분 단위 **실시간 현황**입니다. 일별로 확정된 통계가 필요하면 전력통계정보시스템(EPSIS) 같은 다른 자료를 따로 확인해야 합니다.
* 이 파일은 우리가 쌓기 시작한 날부터만 존재합니다. 과거 값은 이 API로 받을 수 없습니다. **데이터는 수집을 시작한 날이 곧 시작일**입니다. (과거 자료는 전력거래소 사이트의 파일 내려받기)


---

### 이번에 다루지 않은 것

화면에 보이지만 API가 없는 자료(예: 일별 전력수급실적)는 화면 뒤의 주소를 찾아 받아 오는 방법이 있습니다. 하지만 **비공식이라 언제든 깨질 수 있고** 이번 세션에는 내용이 많아 보류했습니다.
저장소의 `extras/scraping/` 에 이와 관련된 자료들을 그대로 남겨 두었습니다.

수고하셨습니다. 🙌